# Upscale — the ideal input and output size

Every model published so far is pinned to one geometry: 128 LR pixels of step, 4 pixels
of halo at each edge, so a 136×136 input and a 272×272 output of which 256×256 is kept.
That geometry was chosen on desktop CPU timings, before there was a browser measurement
to choose it on, and the browser has since disagreed with it twice — a step-256 export
reached 5 ms per 1080p frame against 8, and a step-320 one 4 ms. Nothing has settled the
question, and the constant is still 128.

This notebook settles it. It is not a model notebook: it trains nothing and publishes no
new architecture. It takes `upscale_web`'s trained weights as a fixed thing to measure
and asks the four questions the deployment has to answer before any of it can be wired
into `src/`:

1. **How big is a tile?** — the step, and the input and output shapes that follow.
2. **How many tiles is a run?** — the batch, which is part of the exported shape.
3. **What upscale factor?** — ×2, ×3 or ×4 changes what a frame costs per pixel more
   than any other decision here.
4. **What survives on a weaker GPU?** — the answer is measured on one Ampere card, and
   the client has to run on the mid and mid-low tier cards most people have.

## The metric is nanoseconds per output pixel, and frames per second is not a column

A run of one of these graphs is a *tile*, and a tile is not a frame. Frames per second
is a rate of runs, so it prices the geometry a graph was exported at and not the model
inside it: at ×2 a 256 step is worth four times a 128 step before either one has been
run, and a whole-frame export is worth forty. Two geometries cannot be compared by it at
all, which is exactly the comparison this notebook is about.

So everything here is charged per output pixel, and a frame is derived at the end rather
than assumed at the start. There are three pixel counts and they are all different:

| | what it is | what it charges for |
| --- | --- | --- |
| **computed** | `batch × (step + 2·halo)² × scale²` | what the GPU actually ran |
| **kept** | `batch × step² × scale²` | what the tile contributes to the picture |
| **delivered** | `width × height` of the frame | what the viewer receives |

`computed / kept` is the halo tax, a property of the architecture and the step.
`kept × runs / delivered` is the tiling tax, what a step wastes off the edge of a frame
it does not divide. **Both are paid, and only the second one is ever forgotten** — the
shipped 128 step keeps a 256 pixel tile, and 1080 rows need five of them: 1280 rows
computed to deliver 1080, 18% of the frame thrown away before the halo is counted. No
measurement of a single tile, anywhere on the benchmark page, can see that.

`ns / delivered pixel` is the number a client is really billed. A frame rate falls out of
it whenever one is wanted, and at 1080p60 the budget is **8.0 ns per pixel** — 16.7 ms
over 2.07 M pixels — with the decode, the compositing and the rest of the browser still
to come out of it.

In [ ]:
"""Configuration - every knob this notebook has."""

from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints")

# The sweep's own graphs. Dozens of them, one per geometry, and none of them a model
# anybody would publish - they are the same weights at different shapes. They stay out of
# `benchmark/www/models/`, which only the chosen family is published into at the end.
SWEEP_DIR = CHECKPOINT_DIR / "geometry"

# The steps the sweep walks, in LR pixels kept per tile.
STEPS = (32, 64, 96, 128, 160, 192, 256, 320, 384, 512)

# Batches, and the steps they are swept at. A batch is part of the exported shape, not a
# way of calling a graph: a static shape is the only thing graph capture will record.
BATCHES = (1, 2, 4, 8, 16)
BATCH_STEPS = (64, 128)

# Upscale factors. The dataset is x2, so x3 and x4 are timed rather than scored - the
# arithmetic and the shapes are exact at any factor, the weights are not.
SCALES = (2, 3, 4)

# The outputs a client would be asked for. 1080p is the one every table below defaults to.
FRAMES = {
    "720p": (1280, 720),
    "1080p": (1920, 1080),
    "1440p": (2560, 1440),
    "4K": (3840, 2160),
}
FRAME = FRAMES["1080p"]

# 16.7 ms over the pixels of a 1080p frame. Every per-pixel number below is read against
# this, and it is the whole budget - decode, compositing and the page itself come out of
# the same 16.7 ms.
BUDGET_NS_PER_PIXEL = 16.7e6 / (FRAME[0] * FRAME[1])

# The browser sweep: every graph the last cell publishes, run on the benchmark page on an
# NVIDIA Ampere card in Chromium - fp16, WebGPU, GPU-resident tensors, graph capture, the
# median column, one pass each at a one second sample. This is the measurement the whole
# notebook is about; everything measured on the CPU below is here for the shape of a curve
# and for nothing else.
#
# One pass each rather than three interleaved rounds, so read differences under 10% as
# noise - the page's own note says the same configuration drifts that far on this card.
# The older sweep in `summary.md` read the same shapes about 30% faster on a quieter
# machine, which moves every row together and no ratio between them.
# The same card read about twice as fast in the sweep recorded in `summary.md`, taken
# months earlier on a machine that was not also running a browser under automation. Both
# are kept: every row moves together between them, so the ratios this notebook argues from
# are the same in both, and the absolute per-pixel cost of this model on this card is a
# band rather than a number.
SUMMARY_TILE_MS = {64: 0.155, 128: 0.202, 192: 0.311, 256: 0.452, 320: 0.657}

BROWSER_MS = [
    {"scale": 2, "step": (64, 64), "batch": 1, "ms": 0.186},
    {"scale": 2, "step": (128, 128), "batch": 1, "ms": 0.293},
    {"scale": 2, "step": (192, 192), "batch": 1, "ms": 0.516},
    {"scale": 2, "step": (256, 256), "batch": 1, "ms": 0.893},
    {"scale": 2, "step": (384, 384), "batch": 1, "ms": 1.721},
    # The exact 1080p tilings, which are the point of the rectangle.
    {"scale": 2, "step": (135, 192), "batch": 1, "ms": 0.420},
    {"scale": 2, "step": (270, 192), "batch": 1, "ms": 0.789},
    {"scale": 2, "step": (180, 320), "batch": 1, "ms": 0.755},
    {"scale": 2, "step": (270, 320), "batch": 1, "ms": 1.175},
    # Batches of the 128 tile, and the whole frame in one run.
    {"scale": 2, "step": (128, 128), "batch": 4, "ms": 0.850},
    {"scale": 2, "step": (128, 128), "batch": 16, "ms": 3.040},
    {"scale": 2, "step": (540, 960), "batch": 1, "ms": 5.703},
    # The same 128 tile upscaled harder.
    {"scale": 3, "step": (128, 128), "batch": 1, "ms": 0.446},
    {"scale": 4, "step": (128, 128), "batch": 1, "ms": 0.463},
]

# Nominal vendor figures - fp32 TFLOPS and GB/s - for the cards the client has to run on.
# Order-of-magnitude, used for ratios and floors and for nothing else: a real number for
# any of them has to come off the benchmark page on that card.
GPUS = {
    "Iris Xe 96EU":   {"tflops": 1.7, "bandwidth": 60, "tier": "mid-low, shared memory"},
    "GTX 1650":       {"tflops": 2.9, "bandwidth": 128, "tier": "mid-low"},
    "Apple M1 8-core": {"tflops": 2.6, "bandwidth": 68, "tier": "mid-low, shared memory"},
    "RTX 3050 laptop": {"tflops": 5.5, "bandwidth": 192, "tier": "mid"},
    "RX 6600":        {"tflops": 8.9, "bandwidth": 224, "tier": "mid"},
    "RTX 3060":       {"tflops": 12.7, "bandwidth": 360, "tier": "mid-high (the reference)"},
}
# `summary.md` says only "an NVIDIA Ampere card". A 3060 is assumed so that the other
# cards can be scaled against something, and every scaled number below moves with that
# assumption - it is a ratio between two nominal figures, not a measurement of either.
REFERENCE_GPU = "RTX 3060"

# WebGPU's default limits. A tile is not free to grow past them: the runtime refuses a
# session whose buffers exceed what the device was created with, and the defaults are what
# a page gets unless it asks for more - which a mid-low card may not have to give.
MAX_BUFFER_BYTES = 256 * 1024 * 1024
MAX_BINDING_BYTES = 128 * 1024 * 1024

# Timing. Every sample is a block of runs timed together and divided, and the median of
# `REPEATS` blocks is reported - the same discipline the browser page uses, for the same
# reason.
SAMPLE_SECONDS = 0.5
REPEATS = 3
WARMUP = 3

# The family published to the benchmark page at the end, for the browser to measure on a
# real GPU. Steps at batch 1, then batches at one step, then the exact tilings.
PUBLISH_STEPS = (64, 128, 192, 256, 384)
PUBLISH_BATCH_STEP = 128
PUBLISH_BATCHES = (4, 16)
PUBLISH_SCALES = (3, 4)

# Exact tilings of a 1080p frame, as (kept height, kept width) - the geometries no square
# step can express, and the ones the argument ends on. A ladder rather than one, because
# "exact" and "large enough" are two conditions and the page has to price both.
PUBLISH_EXACT_KEPT = ((270, 384), (540, 384), (360, 640), (540, 640))

In [ ]:
"""Imports, and the trained weights this notebook measures rather than produces."""

import functools
import itertools
import math
import time

import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import webexport

SWEEP_DIR.mkdir(parents=True, exist_ok=True)

# The measurement here is a CPU one, and it is used for the shape of a curve rather than
# for any absolute number - see the section that fits it. The browser page is where the
# numbers that decide anything come from.
print("torch       ", torch.__version__)
print("onnxruntime ", ort.__version__, ort.get_available_providers())
print("threads     ", ort.SessionOptions().intra_op_num_threads or "default (all cores)")
print(f"frame budget {BUDGET_NS_PER_PIXEL:.2f} ns per output pixel at "
      f"{FRAME[0]}x{FRAME[1]}, 60 fps")

In [ ]:
"""The model whose shape is being swept: `upscale_web`'s architecture and its weights.

Restated here rather than imported, because a notebook is one file that runs - but the
weights are loaded from the checkpoint that notebook wrote, so this is that model and not
a lookalike. Nothing is trained here. Timing does not depend on what the weights are, only
on how many there are and what shape they run at, which is why the x3 and x4 variants
below are honest about speed while saying nothing about quality.
"""

def bicubic_kernel(scale, size=5):
    """The bicubic upsampling filter as `scale^2` kernels, one per output phase.

    Recovered by pushing a unit impulse through `F.interpolate` at every position of the
    window, the same way `upscale_web` does it: the operator is linear, so its weights can
    be read out of it instead of derived from a cubic formula whose `a` would have to be
    guessed. It works at any scale, which is what lets the x3 and x4 variants exist.
    """
    impulse = torch.zeros(1, 1, size, size)
    weights = torch.zeros(scale * scale, size, size)
    centre = size // 2
    for y in range(size):
        for x in range(size):
            impulse.zero_()
            impulse[0, 0, y, x] = 1.0
            out = F.interpolate(impulse, scale_factor=scale, mode="bicubic",
                                align_corners=False)[0, 0]
            for dy in range(scale):
                for dx in range(scale):
                    weights[dy * scale + dx, y, x] = out[centre * scale + dy,
                                                         centre * scale + dx]
    return weights


class BicubicSkip(nn.Module):
    """Bicubic as one convolution, so the graph carries no Resize."""

    def __init__(self, scale, size=5):
        super().__init__()
        self.conv = nn.Conv2d(3, 3 * scale * scale, size, padding=size // 2, bias=False)
        phases = bicubic_kernel(scale, size)
        weight = torch.zeros_like(self.conv.weight)
        for channel in range(3):
            for phase in range(scale * scale):
                weight[channel * scale * scale + phase, channel] = phases[phase]
        with torch.no_grad():
            self.conv.weight.copy_(weight)
        self.conv.weight.requires_grad_(False)

    def forward(self, x):
        return self.conv(x)


class WebUpscaler(nn.Module):
    def __init__(self, widths=(32, 16), stem=5, skip=5, scale=2):
        super().__init__()
        self.scale = scale
        self.skip = BicubicSkip(scale, skip)

        layers = []
        width = 3
        for index, out in enumerate(widths):
            kernel = stem if index == 0 else 3
            layers += [nn.Conv2d(width, out, kernel, padding=kernel // 2),
                       nn.ReLU(inplace=True)]
            width = out
        self.body = nn.Sequential(*layers)
        self.tail = nn.Conv2d(width, 3 * scale * scale, 3, padding=1)
        self.shuffle = nn.PixelShuffle(scale)

        # One pixel per 3x3 in the path, stem//2 for the first, and the skip's own reach.
        # It does not depend on the scale: every convolution runs at LR.
        self.halo = max(stem // 2 + len(widths), skip // 2)

    def forward(self, x):
        return self.shuffle(self.skip(x) + self.tail(self.body(x)))


def build(scale=2):
    """The model at a scale, with `upscale_web`'s weights wherever they still fit.

    At x2 every tensor is loaded and this is exactly the published model. At x3 and x4 the
    skip is exact - it is derived, not trained - and the tail is a different shape, so it
    keeps its zero initialisation: an untrained model that computes bicubic. That is the
    right thing to time and the wrong thing to score, and no dB is claimed for it here.
    """
    checkpoint = torch.load(CHECKPOINT_DIR / "upscale_web.pt", map_location="cpu",
                            weights_only=True)
    model = WebUpscaler(tuple(checkpoint["widths"]), checkpoint["stem"],
                        checkpoint["skip"], scale)
    state = {key: value for key, value in checkpoint["state_dict"].items()
             if key in model.state_dict()
             and model.state_dict()[key].shape == value.shape}
    # The skip is derived rather than trained - `bicubic_kernel` writes it at any scale -
    # so a skip tensor the checkpoint could not supply is not a tensor that is missing.
    missing = [key for key in model.state_dict()
               if key not in state and not key.startswith("skip.")]
    model.load_state_dict(state, strict=False)
    model.eval()
    return model, checkpoint, missing


model, checkpoint, missing = build(2)
HALO = model.halo
assert not missing, f"the x2 model should load whole, but {missing} did not"
print(f"{checkpoint['architecture']}  x{checkpoint['scale']}  widths "
      f"{checkpoint['widths']}  halo {HALO}  "
      f"{sum(p.numel() for p in model.parameters()):,} parameters")
print(f"trained {checkpoint['trained']}, {checkpoint['val_psnr']:.2f} dB on the "
      f"validation split")

for scale in SCALES:
    variant, _, absent = build(scale)
    with torch.no_grad():
        shape = tuple(variant(torch.zeros(1, 3, 16, 16)).shape)
    print(f"  x{scale}: {shape}, halo {variant.halo}, "
          + ("every weight loaded" if not absent
             else f"skip derived exactly, {len(absent)} tail tensors untrained "
                  f"(timing only)"))

## 1. The three sizes, and the fourth one nobody counts

A tile has three sizes and confusing them is the usual bug — `summary.md` already says
so. The fourth is the frame it is tiled over, and that one is not a property of the tile
at all, which is why no measurement of a tile can see it.

```
  step          what the tile contributes to the output           128 LR / 256 HR
  model input   step plus the halo at both edges                  136 LR
  kept          the centre, after the halo is cropped             256 HR
  frame         what the tiling of a whole picture computes       5 rows of 256 for 1080
```

The halo tax falls as the step grows — at halo 4 a 128 step computes 13% more pixels than
it keeps, a 256 step 6%, a 512 step 3%. The tiling tax does the opposite: the larger the
tile, the more of it hangs off the edge of a frame that does not divide by it. They cross,
and where they cross is the answer this section is after.

In [ ]:
"""The three sizes of a tile, and what a frame of them costs."""

# The geometry lives in `webexport.py` rather than here: the exporter turns it into a
# shape, this notebook sweeps it, and whatever drives inference in `src/` will need the
# same rule. These are those functions with this model's halo already in them.
geometry = functools.partial(webexport.geometry, halo=HALO)
tiling = webexport.tiling
exact_tilings = functools.partial(webexport.exact_tilings, halo=HALO)
plan_tiling = functools.partial(webexport.plan_tiling, halo=HALO)


def shape_text(pair):
    """A (height, width) shape the way a person reads it."""
    return f"{pair[1]}x{pair[0]}"


print(f"{'step':>6} {'input':>12} {'output':>12} {'kept':>12} {'halo tax':>9} "
      f"{'tiles':>6} {'tiling tax':>11} {'total':>7}")
for step in STEPS:
    shape = geometry(step)
    cover = tiling(FRAME, shape["kept_hr"])
    total = shape["halo_tax"] * cover["tiling_tax"]
    print(f"{step:>6} {shape_text(shape['input'][2:]):>12} "
          f"{shape_text(shape['output'][2:]):>12} {shape_text(shape['kept_hr']):>12} "
          f"{shape['halo_tax']:>8.3f}x {cover['tiles']:>6} "
          f"{cover['tiling_tax']:>10.3f}x {total:>6.3f}x")

print("\nTotal is what a 1080p frame computes for every pixel it delivers. The two taxes "
      "pull opposite ways, so the minimum of the product is neither one's minimum - and "
      "the tiling tax never appears in a measurement of a single tile.")

## 2. A tile does not have to be square, and the good ones are not

The table above pays a tiling tax at every step because 1920×1080 has no square tile that
divides it. It cannot: the kept size would have to divide both 1920 and 1080, so it would
have to divide their gcd of 120 — a 120 HR tile is a 60 LR step, small enough that the
per-run overhead is most of what a frame costs.

Nothing requires a square tile. A convolution does not care, ONNX does not care, and
WebGPU does not care. **A tile whose width divides the frame's width and whose height
divides its height wastes nothing**, and at 1080p there are plenty: 384×270 (5×4 tiles),
640×360 (3×3), 480×540 (4×2), 640×540 (3×2). The step is the kept size over the scale, so
it needs the kept size to be a multiple of the scale as well — which every one of those
is. One of them is worth noticing before any of this is measured: **640×360 divides every
16:9 resolution** — 720p is 2×2 of it, 1080p 3×3, 1440p 4×4, 4K 6×6 — so a client that
picks it can hold one session for every stream it will ever be handed.

This is what `plan_tiling` in `webexport.py` does: given a frame, a scale and a halo, it
picks the exact tiling closest to a target tile area, and says so when the frame has none.

In [ ]:
"""Every exact tiling of the frames a client would be asked for."""

for name, frame in FRAMES.items():
    options = exact_tilings(frame)
    near = [row for row in options if 128 ** 2 <= row["kept_px"] <= 640 ** 2]
    print(f"{name:>6} {frame[0]}x{frame[1]}: {len(options)} exact tilings, "
          f"{len(near)} of a usable size, largest five:")
    for row in near[-5:]:
        print(f"        keeps {shape_text(row['kept_hr']):>9}  "
              f"step {shape_text(row['step']):>9}  "
              f"input {shape_text(row['input'][2:]):>9}  "
              f"{row['tiles']:>3} tiles   halo tax {row['halo_tax']:.3f}x")

print()
for name, frame in FRAMES.items():
    plan = plan_tiling(frame)
    print(f"{name:>6}: keeps {shape_text(plan['kept_hr']):>9} in "
          f"{plan['cols']}x{plan['rows']} = {plan['tiles']:>3} tiles, "
          f"input {shape_text(plan['input'][2:]):>9}, "
          f"tiling tax {plan['tiling_tax']:.3f}x, halo tax {plan['halo_tax']:.3f}x"
          f"{'   (padded: this frame admits no exact tiling near the target)' if plan['padded'] else ''}")

### Changing the tile must not change the picture

All of this is only allowed because tiled inference is *exact*: with a halo at least as
wide as the receptive field, a frame assembled from tiles is bit-identical to the same
frame run whole, and no seam blending is needed. `summary.md` establishes that for the
128 step; it is a property of the halo rather than of the step, so it holds at every
geometry here — but it is cheap to check, and a check is worth more than a citation when
the whole notebook is about moving that geometry around.

In [ ]:
"""Tiled against whole-frame, at the halo the model declares and one pixel short of it."""

def upscale_tiled(model, frame_lr, step, halo, scale=2):
    """Run an LR frame tile by tile and reassemble the kept centres."""
    step_h, step_w = webexport.as_size(step)
    padded = F.pad(frame_lr, (halo, halo, halo, halo), mode="replicate")
    height, width = frame_lr.shape[2], frame_lr.shape[3]
    out = torch.zeros(1, 3, height * scale, width * scale)
    for top in range(0, height, step_h):
        for left in range(0, width, step_w):
            tile = padded[:, :, top:top + step_h + 2 * halo,
                          left:left + step_w + 2 * halo]
            with torch.no_grad():
                result = model(tile)
            keep = result[:, :, halo * scale:(halo + step_h) * scale,
                          halo * scale:(halo + step_w) * scale]
            out[:, :, top * scale:(top + step_h) * scale,
                left * scale:(left + step_w) * scale] = keep
    return out


plan = plan_tiling(FRAME)
step_h, step_w = plan["step"]
frame_lr = torch.rand(1, 3, step_h * 2, step_w * 2)
with torch.no_grad():
    whole = model(frame_lr)

# The true frame border is where a whole-frame run and a tiled one are always allowed to
# differ - one pads with zeros after every layer, the other with real pixels at the first -
# so the comparison is over the interior, which is the part tiling is a claim about.
edge = 2 * HALO * plan["scale"]
for halo in (HALO, HALO - 1):
    tiled = upscale_tiled(model, frame_lr, (step_h, step_w), halo)
    difference = float((tiled[..., edge:-edge, edge:-edge]
                        - whole[..., edge:-edge, edge:-edge]).abs().max())
    verdict = "identical" if difference < 1e-5 else "SEAMS - this halo is too narrow"
    print(f"halo {halo}: max abs difference {difference:.2e} over the interior   {verdict}")

print(f"\nSo the tile size is free to move: {step_w}x{step_h} tiles of a "
      f"{step_w * 2}x{step_h * 2} frame reconstruct it exactly, and every geometry below "
      f"is a speed question rather than a quality one.")

## 3. What a run costs: a fixed part and a part that scales

A run of a graph is one `session.run`, and it costs a fixed amount plus an amount
proportional to the pixels it computes:

```
  ms_per_run  =  a  +  b · computed_pixels
```

`a` is everything that happens once per run whatever the tile is — JavaScript issuing the
call, ONNX Runtime binding, one `queue.submit`, and on the browser page the fence that
settles it. `b` is the model: arithmetic and memory traffic per pixel. The whole question
of tile size is the ratio between them, because

```
  ns per delivered pixel  =  (a + b · computed) · runs · 10⁶ / (width · height)
```

and `runs` falls as the tile grows while `computed` per run rises. **A big `a` wants big
tiles; a big `b` wants tiles that waste nothing.** A weaker GPU is exactly a bigger `b`,
which is why the answer for a mid-low card is not the answer for the card the numbers
were taken on.

In [ ]:
"""Time a graph through ONNX Runtime, the same discipline the browser page uses."""

def session_for(model, shape, batch=1, scale=2, halo=None, dtype=np.float32,
                providers=("CPUExecutionProvider",)):
    """Export this geometry and open a session on it, reusing a graph already written."""
    halo = HALO if halo is None else halo
    step_h, step_w = webexport.as_size(shape)
    name = f"geom_x{scale}_s{step_w}x{step_h}_b{batch}.onnx"
    path = SWEEP_DIR / name
    if not path.exists():
        webexport.export(model, name, (step_h + 2 * halo, step_w + 2 * halo),
                         label=f"geometry sweep x{scale} {step_w}x{step_h} batch {batch}",
                         halo=halo, batch=batch, models_dir=SWEEP_DIR)
    options = ort.SessionOptions()
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(path), options, providers=list(providers))
    feed = {session.get_inputs()[0].name:
            np.random.rand(batch, 3, step_h + 2 * halo, step_w + 2 * halo).astype(dtype)}
    return session, feed, path


def time_session(session, feed, seconds=SAMPLE_SECONDS, repeats=REPEATS, warmup=WARMUP):
    """Milliseconds per run: the median of `repeats` blocks, each filling `seconds`.

    A block, not a run. One run of a small tile finishes inside the noise of the clock and
    of the scheduler, so runs are timed together and divided - and blocks are repeated and
    the median taken, because a laptop under a browser is not a quiet machine.
    """
    for _ in range(warmup):
        session.run(None, feed)
    samples = []
    for _ in range(repeats):
        runs, started = 0, time.perf_counter()
        while time.perf_counter() - started < seconds or runs < 3:
            session.run(None, feed)
            runs += 1
        samples.append((time.perf_counter() - started) * 1000 / runs)
    return sorted(samples)[len(samples) // 2]


def measure(model, step, batch=1, scale=2, frame=FRAME, **kwargs):
    """One row of every sweep below: a geometry, its run time, and its three per-pixel costs."""
    shape = geometry(step, scale=scale, batch=batch)
    session, feed, path = session_for(model, step, batch, scale, **kwargs)
    milliseconds = time_session(session, feed)
    cover = tiling(frame, shape["kept_hr"], batch)
    return {
        **shape,
        "ms": milliseconds,
        "kb": round(path.stat().st_size / 1024, 1),
        # The model with the geometry divided out.
        "ns_per_kept": milliseconds * 1e6 / shape["kept"],
        # The model with the geometry put back in, for one whole frame.
        "ns_per_delivered": milliseconds * cover["runs"] * 1e6 / cover["delivered"],
        "frame_ms": milliseconds * cover["runs"],
        "runs": cover["runs"],
        "tiles": cover["tiles"],
        "tiling_tax": cover["tiling_tax"],
    }

In [ ]:
"""The step sweep: one tile, every size, fp32 on the CPU provider."""

sweep = [measure(model, step) for step in STEPS]

print(f"{'step':>6} {'input':>9} {'ms/run':>9} {'ns/kept px':>11} {'ns/1080p px':>12} "
      f"{'runs':>5} {'frame ms':>9} {'total tax':>10}")
for row in sweep:
    print(f"{row['step'][0]:>6} {row['input'][2]:>4}^2  {row['ms']:>9.3f} "
          f"{row['ns_per_kept']:>11.2f} {row['ns_per_delivered']:>12.2f} "
          f"{row['runs']:>5} {row['frame_ms']:>8.1f}  "
          f"{row['halo_tax'] * row['tiling_tax']:>9.3f}x")

best_kept = min(sweep, key=lambda row: row["ns_per_kept"])
best_delivered = min(sweep, key=lambda row: row["ns_per_delivered"])
print(f"\ncheapest per kept pixel:      step {best_kept['step'][0]} at "
      f"{best_kept['ns_per_kept']:.2f} ns")
print(f"cheapest per delivered pixel: step {best_delivered['step'][0]} at "
      f"{best_delivered['ns_per_delivered']:.2f} ns")
print(f"the two disagree by {abs(best_kept['step'][0] - best_delivered['step'][0])} "
      f"pixels of step, which is the tiling tax having its say")

In [ ]:
"""Fit `ms = a + b x computed`, on this CPU and on the browser numbers from summary.md."""

def machine_ms(shape, machine):
    """What one run of this geometry costs on a fitted machine."""
    return machine["fixed_ms"] + machine["ns_per_computed"] * shape["computed"] / 1e6


def fit(pairs):
    """Least squares through (computed pixels, milliseconds), as (fixed ms, ns per px)."""
    x = np.array([pixels for pixels, _ in pairs], dtype=np.float64)
    y = np.array([milliseconds for _, milliseconds in pairs], dtype=np.float64)
    slope, intercept = np.polyfit(x, y, 1)
    predicted = slope * x + intercept
    residual = float(np.sqrt(np.mean((y - predicted) ** 2)))
    return {"fixed_ms": float(intercept), "ns_per_computed": float(slope) * 1e6,
            "rmse_ms": residual, "n": len(pairs)}


cpu_fit = fit([(row["computed"], row["ms"]) for row in sweep])

# The card, out of the browser sweep: every x2 batch-1 row of it, from a 64 step to a whole
# 1080p frame in one run - a factor of 250 in area, which is what makes two parameters worth
# fitting. The computed pixel count comes from `geometry` rather than from the table, so a
# rectangular tile and a square one enter the same fit on the same terms.
browser = [{**row, **geometry(row["step"], scale=row["scale"], batch=row["batch"])}
           for row in BROWSER_MS]
ampere_fit = fit([(row["computed"], row["ms"]) for row in browser
                  if row["scale"] == 2 and row["batch"] == 1])

# The same card, months earlier, from summary.md - fitted the same way over square steps.
quiet_fit = fit([(geometry(step)["computed"], milliseconds)
                 for step, milliseconds in SUMMARY_TILE_MS.items()])

for name, model_fit in (("this CPU, fp32", cpu_fit), ("Ampere, this sweep", ampere_fit),
                        ("Ampere, summary.md", quiet_fit)):
    print(f"{name:<22} fixed {model_fit['fixed_ms']:.4f} ms per run, "
          f"{model_fit['ns_per_computed']:.4f} ns per computed pixel, "
          f"rmse {model_fit['rmse_ms']:.4f} ms")

# The page measures a bare dispatch - session.run with the fence deliberately skipped - at
# 0.046 ms. The fit says a run costs more than twice that before it computes anything, so
# roughly half of the fixed cost is the submit and the rest is everything else that happens
# once a run: binding, the runtime's own JavaScript, and this batch's share of the fence.
spread = ampere_fit["ns_per_computed"] / quiet_fit["ns_per_computed"]
print(f"\nThe two Ampere fits are the same card {spread:.1f}x apart, which is a fact about "
      f"the machine on the day and not about any geometry: every row moves together, so "
      f"every ratio below survives it and every absolute ns/px is a band this wide.")

crossover = ampere_fit["fixed_ms"] * 1e6 / ampere_fit["ns_per_computed"]
print(f"\nOn the Ampere card the fixed cost of a run buys {crossover:,.0f} computed "
      f"pixels, which is a tile of {math.sqrt(crossover) / 2:.0f} LR pixels of step.")
print("Below that step a run is mostly overhead; above it the tile is mostly work. That "
      "one number is what the whole question turns on.")

### What the fit says, and why the two disagree

The two fits are the same model of two very different machines, and the interesting
quantity is `a / b` — the computed pixels one run's fixed cost is worth. Below that tile
size a run is mostly overhead, above it the tile is mostly work, and the optimum sits
just past it once the tiling tax is added.

The CPU provider has a small `a` (no queue, no fence, no JavaScript) and a large `b`
(twenty threads against thousands of shaders), so it wants small tiles. WebGPU in a
browser is the opposite on both counts, and it is the one that decides — which is the
whole reason `summary.md` says the CPU number and the browser number disagree about which
operators are expensive. This notebook measures the CPU because it is what the machine
has; it *fits* the browser because that is where the client runs.

In [ ]:
"""The browser sweep itself: what each published geometry actually cost on the card."""

for row in browser:
    cover = tiling(FRAME, row["kept_hr"], row["batch"])
    row["ns_per_kept"] = row["ms"] * 1e6 / row["kept"]
    row["ns_per_delivered"] = row["ms"] * cover["runs"] * 1e6 / cover["delivered"]
    row["frame_ms"] = row["ms"] * cover["runs"]
    row["runs"] = cover["runs"]
    row["exact"] = cover["tiling_tax"] == 1.0
    row["predicted"] = machine_ms(row, ampere_fit)

print(f"{'scale':>5} {'step':>10} {'keeps':>10} {'batch':>6} {'ms/run':>8} {'fitted':>7} "
      f"{'ns/kept':>8} {'ns/frame':>9} {'runs':>5} {'1080p':>9}  exact")
for row in sorted(browser, key=lambda row: (row["scale"], row["ns_per_delivered"])):
    # The fit is over x2 rows and does not transfer to another scale: the body runs at LR
    # whatever the scale, but the tail widens as scale^2 and the shuffle moves that much
    # more data, so neither output pixels nor LR pixels alone price it. The x3 and x4 rows
    # are measured, and the fit over-predicts them by about a factor of two - which is the
    # evidence for that sentence rather than a caveat about it.
    fitted = f"{row['predicted']:.3f}" if row["scale"] == 2 else "n/a"
    print(f"x{row['scale']:<4} {shape_text(row['step']):>10} "
          f"{shape_text(row['kept_hr']):>10} {row['batch']:>6} {row['ms']:>8.3f} "
          f"{fitted:>7} {row['ns_per_kept']:>8.2f} {row['ns_per_delivered']:>9.2f} "
          f"{row['runs']:>5} {row['frame_ms']:>7.1f} ms  {'yes' if row['exact'] else ''}")

best = min((row for row in browser if row["scale"] == 2 and row["runs"] > 1),
           key=lambda row: row["ns_per_delivered"])
whole_frame = min((row for row in browser if row["runs"] == 1),
                  key=lambda row: row["ns_per_delivered"])
print(f"\nThe cheapest tiling measured is {shape_text(best['kept_hr'])} at "
      f"{best['ns_per_delivered']:.2f} ns per delivered pixel - "
      f"{best['frame_ms']:.1f} ms a frame in {best['runs']} runs - against "
      f"{whole_frame['ns_per_delivered']:.2f} ns for the whole frame in one.")
print(f"Tiling costs {best['ns_per_delivered'] / whole_frame['ns_per_delivered']:.0%} of "
      f"the one-run cost, and buys {best['runs']} units that can be skipped.")
print("Every exact row has ns/kept equal to its ns/frame, which is what a tiling tax of "
      "exactly 1.0 looks like from the other end.")

In [ ]:
"""The optimum step, as a function of the machine it runs on."""

def frame_cost(step, machine, frame=FRAME, scale=2, batch=1):
    """Nanoseconds per delivered pixel, from a fitted machine rather than from a clock."""
    shape = geometry(step, scale=scale, batch=batch)
    cover = tiling(frame, shape["kept_hr"], batch)
    milliseconds = machine["fixed_ms"] + machine["ns_per_computed"] * shape["computed"] / 1e6
    return milliseconds * cover["runs"] * 1e6 / cover["delivered"]


# A weaker card is the same fixed cost - dispatch is the browser's and the driver's, not
# the shaders' - and proportionally more time per pixel. That is the whole model of a
# mid-low tier GPU used here, and it is deliberately crude.
machines = {"Ampere (measured)": ampere_fit}
reference = GPUS[REFERENCE_GPU]["tflops"]
for name, spec in GPUS.items():
    machines[f"{name} (scaled)"] = {
        "fixed_ms": ampere_fit["fixed_ms"],
        "ns_per_computed": ampere_fit["ns_per_computed"] * reference / spec["tflops"],
    }

# The search runs up to a tile that covers the frame in one run, because that is where the
# curve turns out to end: both taxes fall as the tile grows, and nothing in this model of a
# run pushes back except the sawtooth of a step that does not divide the frame. The ceiling
# on a tile comes from the two sections after this one, not from here.
fine_steps = list(range(32, 561, 8))
planned = plan_tiling(FRAME)

label = shape_text(planned["kept_hr"])
print(f"{'machine':<26} {'best step':>10} {'ns/px':>7} {f'at {label}':>12} "
      f"{'whole frame':>12} {'1080p60':>18}")
for name, machine in machines.items():
    cost, step = min((frame_cost(step, machine), step) for step in fine_steps)
    at_plan = frame_cost(planned["step"], machine)
    at_frame = frame_cost((FRAME[1] // 2, FRAME[0] // 2), machine)
    # The band, not a number: the two Ampere fits of one card are `spread` apart, so every
    # scaled figure here is worth stating at both ends of that.
    print(f"{name:<26} {step:>10} {cost:>7.2f} {at_plan:>12.2f} {at_frame:>12.2f} "
          f"{at_plan / spread / BUDGET_NS_PER_PIXEL:>8.0%} -{at_plan / BUDGET_NS_PER_PIXEL:>7.0%}")

print("\nThe minimum is at the top of the range on every machine, and the column that "
      "matters is the third: what the tiling this notebook recommends costs, against a "
      "whole-frame run that gives up every skipped tile to get there.")

# The curve is flat over most of its range, which matters more than where its minimum is:
# a step chosen for one card should not fall apart on another, and over 128-512 none of
# these machines varies by more than a third.
plt.figure(figsize=(9, 4))
for name, machine in machines.items():
    plt.plot(fine_steps, [frame_cost(step, machine) for step in fine_steps], label=name)
plt.axhline(BUDGET_NS_PER_PIXEL, color="black", linestyle="--", linewidth=1,
            label="1080p60, the whole frame budget")
plt.axvline(planned["step"][1], color="grey", linestyle=":", linewidth=1,
            label=f"{planned['kept_hr'][1]}x{planned['kept_hr'][0]} tile, exact at 1080p")
plt.xlabel("LR step (pixels)")
plt.ylabel("ns per delivered pixel")
plt.title("Cost of a 1080p frame per pixel, by tile size")
plt.yscale("log")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### The curve has no interior minimum, and that is the finding

Both taxes fall as the tile grows and the fixed cost of a run is spread over more pixels,
so on this model of a run **the cheapest way to upscale a frame is to upscale it in one
run** - which is exactly what `webexport.to_frame` already builds, and what the
whole-frame column above prices. Every mid and mid-low card here would rather run one
960x540 tile than forty 136x136 ones.

That is a true statement about a frame in which every pixel changed, and it is the wrong
conclusion for a remote desktop, where most frames change in one small place. The tile
size is not bounded from above by the cost of a run at all. It is bounded by three things
a per-run measurement cannot see:

1. **What can be skipped** - the tile is the unit of "this did not change", and a bigger
   unit skips less. Section 5.
2. **What fits** - WebGPU's default buffer limits, and a mid-low card's memory. Section 6.
3. **What blocks** - one run is one unyielding block of the GPU's time; forty tiles let
   the compositor in forty times, one frame-sized run does not.

So the useful question is not "where is the minimum" but "how large can a tile be before
it stops being skippable", and the answer to that is content, not hardware.

## 4. Batching: the other way to amortise a dispatch

A batch and a bigger tile buy the same thing — fewer runs for the same pixels — and they
pay for it differently:

| | halo | frame edge | dirty-region granularity |
| --- | --- | --- | --- |
| one tile of step `s·√N` | paid once | wasted at the frame edge | coarse |
| `N` tiles of step `s` | paid `N` times | nothing extra | unchanged |

So a bigger tile is the cheaper of the two *per pixel computed*, and a batch is the one
that leaves the rest of the client alone. That matters because the largest optimisation
available on desktop content is not in the model at all: **a tile whose input did not
change has an output that did not change**, and 44% of tiles change per frame at step 128
on a synthesised desktop sequence. Doubling the step quadruples the area a single moved
cursor dirties.

A batch also has a latency: `N` tiles settle together, so the first one is not ready until
the last one is. For a stream that is fine — the frame is not ready until all of it is.

In [ ]:
"""What a batch buys, at two steps, against a single tile of the same total area."""

batch_rows = []
for step, batch in itertools.product(BATCH_STEPS, BATCHES):
    batch_rows.append(measure(model, step, batch))

print(f"{'step':>6} {'batch':>6} {'ms/run':>9} {'ns/kept px':>11} {'vs batch 1':>11} "
      f"{'computed px':>12} {'runs/frame':>11}")
for step in BATCH_STEPS:
    single = next(row for row in batch_rows
                  if row["step"][0] == step and row["batch"] == 1)
    for row in [row for row in batch_rows if row["step"][0] == step]:
        print(f"{step:>6} {row['batch']:>6} {row['ms']:>9.3f} "
              f"{row['ns_per_kept']:>11.2f} "
              f"{single['ns_per_kept'] / row['ns_per_kept']:>10.2f}x "
              f"{row['computed']:>12,} {row['runs']:>11}")
    print()

# The like-for-like comparison: N tiles of step s against one tile that keeps as much.
print(f"{'kept px':>10} {'as batch':>20} {'as one tile':>20} {'computed ratio':>15}")
for step, batch in itertools.product(BATCH_STEPS, (4, 16)):
    batched = geometry(step, batch=batch)
    side = step * int(math.sqrt(batch))
    single = geometry(side)
    assert batched["kept"] == single["kept"]
    print(f"{batched['kept']:>10,} {f'{batch} x {step}':>20} {f'1 x {side}':>20} "
          f"{batched['computed'] / single['computed']:>14.3f}x")
print("\nThe batch computes more, because it pays the halo once per tile. What it does not "
      "do is grow the unit of work a dirty-region client is allowed to skip.")

# The CPU provider has no dispatch worth amortising - its fitted fixed cost is a rounding
# error, and the table above duly shows a batch buying nothing. On the card the fixed cost
# is a third of what a 128 tile costs, so the prediction there is a real saving - and the
# page has been run at batch 4 and 16, so it is a prediction with an answer beside it.
print(f"\n{'batch':>6} {'predicted':>10} {'measured':>9} {'ms per tile':>12} "
      f"{'vs batch 1':>11}   Ampere, step {PUBLISH_BATCH_STEP}")
measured = {row["batch"]: row["ms"] for row in browser
            if row["scale"] == 2 and row["step"] == (PUBLISH_BATCH_STEP,) * 2}
base = measured.get(1)
for batch in BATCHES:
    predicted = machine_ms(geometry(PUBLISH_BATCH_STEP, batch=batch), ampere_fit)
    seen = measured.get(batch)
    per_tile = (seen or predicted) / batch
    print(f"{batch:>6} {predicted:>10.3f} {(f'{seen:.3f}' if seen else '-'):>9} "
          f"{per_tile:>12.3f} {base / per_tile:>10.2f}x"
          + ("" if seen else "   (predicted)"))
print("Measured, a batch of 16 at step 128 costs about two thirds what 16 separate runs "
      "cost, at the same halo, the same tiling and the same skip granularity - less than "
      "the fit predicts, because the fit charges the whole fixed cost to the dispatch and "
      "some of it is per-tile work the batch pays anyway.")

## 5. The dirty region decides more than the tile does

On a desktop stream most of the picture is unchanged most of the time, and a tile whose
input did not change does not have to be run at all. That makes the useful cost of a frame
the cost of the tiles that *moved*, and the number of those depends on the tile size in
the opposite direction to everything above: a 32×32 cursor moving across the screen dirties
one 256 tile or four 128 tiles, but the four 128 tiles are a quarter of the pixels each.

So the optimum step for a full-screen video and the optimum step for a moving cursor are
not the same step, and neither is a number the model has any say in.

In [ ]:
"""What a frame costs when only part of it moved, by tile size and by what moved."""

DIRTY_STEPS = (64, 128, 192, 256, 320, 384)

CONTENT = {
    "cursor, 32x32": (32, 32),
    "a line of text, 400x30": (400, 30),
    "a dragged window, 800x600": (800, 600),
    "video, 1280x720": (1280, 720),
    "everything": FRAME,
}


def dirty_runs(frame, kept_hr, region, batch=1):
    """Runs needed for one moved rectangle, averaged over where it happens to sit.

    A `w x h` region at an arbitrary offset touches ceil(w / kept) or one more columns
    depending on alignment, and the average is w / kept + 1 - which is the expression
    below, clamped to the tiling and rounded into whole runs.
    """
    kept_h, kept_w = kept_hr
    cover = tiling(frame, kept_hr, batch)
    cols = min(cover["cols"], region[0] / kept_w + 1)
    rows = min(cover["rows"], region[1] / kept_h + 1)
    return math.ceil(cols * rows / batch)


print(f"{'content':<26}" + "".join(f"{step:>9}" for step in DIRTY_STEPS))
for name, region in CONTENT.items():
    cells = []
    for step in DIRTY_STEPS:
        shape = geometry(step)
        runs = dirty_runs(FRAME, shape["kept_hr"], region)
        milliseconds = (ampere_fit["fixed_ms"]
                        + ampere_fit["ns_per_computed"] * shape["computed"] / 1e6) * runs
        cells.append(f"{milliseconds:>8.2f}")
    print(f"{name:<26}" + "".join(cells) + "   ms per frame")

print("\nEvery column is the same model on the same card. The best tile size for a cursor "
      "and the best one for a full frame are three doublings apart, and a client that has "
      "to pick one shape picks against one of them.")


# The one number that decides whether tiling is worth doing at all. A whole-frame run pays
# one fixed cost and computes every pixel; a tiling pays one per tile and computes only the
# tiles that moved. Below this fraction of moved tiles the tiling wins, above it the frame
# does - and it is a property of the geometry and the card, not of the model.
whole = geometry((FRAME[1] // 2, FRAME[0] // 2))
tile = geometry(planned["step"])
break_even = machine_ms(whole, ampere_fit) / (planned["runs"] * machine_ms(tile, ampere_fit))
print(f"\nOne 1080p frame whole: {machine_ms(whole, ampere_fit):.2f} ms. "
      f"All {planned['runs']} tiles of {shape_text(planned['kept_hr'])}: "
      f"{planned['runs'] * machine_ms(tile, ampere_fit):.2f} ms.")
print(f"So tiling this frame is worth it below {break_even:.0%} of tiles moving, and a "
      f"whole-frame run is worth it above.")
print("44% of tiles moved per frame on the synthesised desktop sequence in upscale_web, "
      "comfortably inside that: on desktop content tiling wins, and full-motion video is "
      "the case for running the frame whole. A client should hold both graphs and pick by "
      "how much of the frame is dirty - they are 20 KB each.")

## 6. What a mid and mid-low tier GPU can actually hold and run

Two limits have nothing to do with speed, and both of them bite before a tile gets large:

- **WebGPU's default buffer limits.** A session's tensors are storage buffers, and the
  default `maxStorageBufferBindingSize` is 128 MiB with `maxBufferSize` at 256 MiB. A page
  may ask for more and a mid-low card may not have it to give.
- **What the widest activation costs.** The body runs at LR with 32 channels, so the
  largest single tensor is `batch × 32 × (step + 2·halo)² × bytes` — four times the input
  at fp32, and it grows with the batch exactly as fast as with the area.

The arithmetic floor is the other half. The model is 9,636 MACs per LR pixel, so per
*output* pixel at ×2 it is 2,409 MACs — and at ×3 it is 1,071, and at ×4 it is 602,
because the body always runs at LR. That is the strongest argument in this notebook for a
larger scale factor, and it is arithmetic rather than measurement.

In [ ]:
"""Memory per run, against the limits, and the arithmetic floor per card."""

BYTES = {"float32": 4, "float16": 2}


def working_set(step, batch=1, scale=2, widest=32, dtype="float16"):
    """The tensors one run needs at once, and the largest single one of them."""
    shape = geometry(step, scale=scale, batch=batch)
    size = BYTES[dtype]
    input_bytes = batch * 3 * shape["input"][2] * shape["input"][3] * size
    output_bytes = batch * 3 * shape["output"][2] * shape["output"][3] * size
    body_bytes = batch * widest * shape["input"][2] * shape["input"][3] * size
    # The tail feeds the shuffle at LR with 3*scale^2 channels, which is the other big one.
    tail_bytes = batch * 3 * scale * scale * shape["input"][2] * shape["input"][3] * size
    return {"largest": max(input_bytes, output_bytes, body_bytes, tail_bytes),
            "total": input_bytes + output_bytes + body_bytes + tail_bytes}


print(f"{'step':>6} {'batch':>6} {'largest tensor':>16} {'working set':>13}  fp16, x2")
for step, batch in itertools.product((128, 256, 512), (1, 4, 16)):
    memory = working_set(step, batch)
    flags = []
    if memory["largest"] > MAX_BINDING_BYTES:
        flags.append("over the 128 MiB binding limit")
    if memory["total"] > MAX_BUFFER_BYTES:
        flags.append("over the 256 MiB buffer limit")
    print(f"{step:>6} {batch:>6} {memory['largest'] / 2**20:>13.1f} MiB "
          f"{memory['total'] / 2**20:>10.1f} MiB  {', '.join(flags)}")


def macs_per_lr_pixel(model, size=32):
    """Counted from the convolutions that run, not from the parameters."""
    total = 0

    def hook(module, inputs, output):
        nonlocal total
        kernel = module.kernel_size[0] * module.kernel_size[1]
        pixels = output.shape[2] * output.shape[3]
        total += (module.in_channels // module.groups) * module.out_channels * kernel * pixels

    handles = [m.register_forward_hook(hook) for m in model.modules()
               if isinstance(m, nn.Conv2d)]
    with torch.no_grad():
        model(torch.zeros(1, 3, size, size))
    for handle in handles:
        handle.remove()
    return total / (size * size)


macs_lr = macs_per_lr_pixel(model)
macs_out = macs_lr / 4
print(f"\n{macs_lr:,.0f} MACs per LR pixel, {macs_out:,.0f} per output pixel at x2")

# What fraction of a card's arithmetic peak this model actually reaches, taken from the one
# card there is a measurement for: the fitted cost per computed pixel against the peak of
# the card that fit came from. It is not assumed, and it is the only honest way to turn a
# vendor TFLOPS figure into an expectation for a convolution stack this small.
reference_floor = 2 * macs_out / (GPUS[REFERENCE_GPU]["tflops"] * 1e12) * 1e9
utilisation = reference_floor / ampere_fit["ns_per_computed"]
print(f"the two fits reach {utilisation * spread:.0%} and {utilisation:.0%} of a "
      f"{REFERENCE_GPU}'s arithmetic peak: {quiet_fit['ns_per_computed']:.3f} and "
      f"{ampere_fit['ns_per_computed']:.3f} ns per computed pixel against a "
      f"{reference_floor:.3f} ns floor")

print(f"\n{'card':<18} {'tier':<26} {'floor ns/px':>12} {'expected ns/px':>18} "
      f"{'1080p60':>18}")
for name, spec in GPUS.items():
    # Two flops to a MAC, and a floor is a floor: no card reaches its peak on a stack this
    # small, so the useful columns are the ones with the observed utilisation in them - at
    # both ends of the band the two fits of one card leave.
    peak = 2 * macs_out / (spec["tflops"] * 1e12) * 1e9
    fast, slow = peak / (utilisation * spread), peak / utilisation
    print(f"{name:<18} {spec['tier']:<26} {peak:>12.3f} {fast:>8.2f} -{slow:>8.2f} "
          f"{fast / BUDGET_NS_PER_PIXEL:>8.0%} -{slow / BUDGET_NS_PER_PIXEL:>7.0%}")

print("\nThe floor is arithmetic alone - no halo, no tiling, no dispatch, no memory. "
      "Everything measured has to sit above it, and the gap between a card's floor and "
      "what the page measures on it is what the geometry is costing.")

## 7. The scale factor is the cheapest lever of all

Every convolution in this architecture runs at LR and the upscale is one `DepthToSpace` at
the end. So the arithmetic per *output* pixel falls as `1 / scale²`, and so does the
bandwidth the stream has to carry — a ×4 model receives a sixteenth of the pixels a ×1
stream would.

What it costs is quality, and this notebook cannot say how much: the dataset is a ×2
degradation, so a ×3 or ×4 model here is an untrained tail on an exact bicubic skip. The
speed below is real (timing does not depend on the values in the weights); the quality is
not measured and no dB is claimed.

In [ ]:
"""Cost per output pixel at x2, x3 and x4, at one step and one batch."""

print(f"{'scale':>6} {'step':>6} {'input':>9} {'output':>11} {'MACs/out px':>12} "
      f"{'ms/run':>9} {'ns/kept px':>11} {'LR pixels a frame':>18}")
for scale in SCALES:
    variant, _, _ = build(scale)
    # The step is held at what a x2 client would send, so the comparison is "the same tile
    # of received pixels, upscaled harder" rather than "a different tile".
    row = measure(variant, 128, batch=1, scale=scale)
    lr_frame = (FRAME[0] // scale) * (FRAME[1] // scale)
    print(f"x{scale:<5} {row['step'][0]:>6} {row['input'][2]:>4}^2  "
          f"{'x'.join(map(str, row['output'][2:])):>11} "
          f"{macs_per_lr_pixel(variant) / scale ** 2:>12,.0f} {row['ms']:>9.3f} "
          f"{row['ns_per_kept']:>11.2f} {lr_frame:>18,}")

print(f"\n{'scale':>6} {'browser ms/run':>15} {'ns/kept px':>11} {'ns/frame px':>12} "
      f"{'1080p frame':>12}   the same step on the card")
for scale in SCALES:
    row = next((row for row in browser if row["scale"] == scale
                and row["step"] == (128, 128) and row["batch"] == 1), None)
    if row is None:
        continue
    print(f"x{scale:<5} {row['ms']:>15.3f} {row['ns_per_kept']:>11.2f} "
          f"{row['ns_per_delivered']:>12.2f} {row['frame_ms']:>9.1f} ms")

print("\nThe MACs column is the argument and the browser table is the answer: at the same "
      "step, upscaling x4 instead of x2 costs half as much per delivered pixel and sends "
      "a quarter of the pixels. The whole cost of a scale factor is quality, and this "
      "notebook has not measured any.")

## 8. Publish the family, and let a real GPU finish the argument

Everything above is a CPU measurement and a two-parameter model fitted to five numbers
from one card. The decision needs the same sweep on the cards the client will run on, and
the benchmark page is what takes it — so the last cell publishes the geometries as
graphs, at fp16, which is what a browser should be running.

The page now reports **ns / kept px** and **ns / frame px** rather than a frame rate, so a
row from a 384 step and a row from a 64 step are directly comparable. Run the family on
the target card, read the two columns, and put the numbers back into `fit` above.

In [ ]:
"""Publish the geometry family into benchmark/www/models/, at fp16."""

published = []


def publish(step, batch=1, scale=2, note=""):
    """One geometry, fp32 for the export and fp16 for the page."""
    step_h, step_w = webexport.as_size(step)
    variant = model if scale == 2 else build(scale)[0]
    label = (f"geom x{scale} step {step_w}x{step_h}"
             + (f" batch {batch}" if batch > 1 else "") + (f" - {note}" if note else ""))
    name = f"upscale_geom_x{scale}_s{step_w:04d}x{step_h:04d}_b{batch:02d}"
    source = webexport.export(variant, f"{name}.onnx",
                              (step_h + 2 * HALO, step_w + 2 * HALO),
                              label=f"{label} fp32", halo=HALO, batch=batch,
                              models_dir=SWEEP_DIR)
    info = webexport.to_float16(source, f"{name}.onnx")
    webexport.write_metadata(info["path"], {"label": f"{label} fp16"})
    shape = geometry((step_h, step_w), scale=scale, batch=batch)
    published.append({**info, "label": label, "kept": shape["kept"],
                      "computed": shape["computed"]})
    return info


for step in PUBLISH_STEPS:
    publish(step)
for batch in PUBLISH_BATCHES:
    publish(PUBLISH_BATCH_STEP, batch=batch)
for scale in PUBLISH_SCALES:
    publish(128, scale=scale)

# The exact tilings of a 1080p frame: the geometries this notebook is arguing for, and the
# ones no square step can express.
for kept in PUBLISH_EXACT_KEPT:
    cover = tiling(FRAME, kept)
    publish((kept[0] // 2, kept[1] // 2),
            note=f"exact 1080p, {cover['cols']}x{cover['rows']} tiles")

print(f"{'file':<44} {'kept px a run':>14} {'halo tax':>9} {'KB':>7}")
for info in published:
    print(f"{info['path'].name:<44} {info['kept']:>14,} "
          f"{info['computed'] / info['kept']:>8.3f}x {info['kb']:>7.1f}")
print(f"\n{len(published)} graphs written to {webexport.MODELS_DIR.resolve()}")
print("Serve the page and run them: uv run upscale/benchmark/main.py")

## 9. The answer

**The tile is 640×360 out, from a 320×180 step and a 328×188 model input.** It is the
cheapest tiling measured on the card — 3.28 ns per delivered pixel, 6.8 ms for a 1080p
frame in 9 runs — and it is the one geometry in the table that needs no second thought at
another resolution: **640×360 divides every 16:9 output exactly.** 720p is 2×2 of it,
1080p 3×3, 1440p 4×4, 4K 6×6. One exported graph, one session, one set of buffers, no
frame edge computed and thrown away at any of them. `webexport.TARGET_KEPT_PX` is that
tile and `plan_tiling` returns it for all four.

What the sweep said, in the order it said it:

- **The 128 step that everything is published at is the wrong size, and by a lot.** It
  costs 5.65 ns per delivered pixel against 3.28 — 11.7 ms a frame against 6.8. A third of
  its per-run cost is the fixed cost of making the run at all, paid 40 times a frame.
- **Square steps lose to rectangles that fit.** The best square, 192, reads 3.50 ns per
  kept pixel and 3.73 per delivered one; the gap between those two numbers is the 6.7% of
  the frame it computes off the edge. Every exact tiling has them equal, which is what a
  tiling tax of 1.0 looks like from the far end, and it is the cleanest confirmation that
  the two columns are measuring what they claim to.
- **Bigger still is better, until it stops being skippable.** The whole frame in one run
  is the cheapest thing here at 2.75 ns, and 640×360 costs 19% more than it while leaving
  9 units that can be skipped when they do not change. That 19% is the price of the
  dirty-region optimisation, and it is the only number that decides whether tiling is
  worth doing at all.
- **Batching works, and less than the fit predicted.** 16 tiles of step 128 in one run
  cost 0.190 ms a tile against 0.293 run singly — 1.54×, where the fit said 2.1×. So some
  of what looked like a per-run cost is per-tile work a batch pays anyway. Batching is
  still the way to amortise a dispatch without coarsening what can be skipped, and a
  client with scattered dirty tiles should use it.
- **Scale is the cheapest lever, and it is unmeasured where it matters.** At the same
  step, ×4 costs 2.68 ns per delivered pixel against ×2's 5.65 and sends a quarter of the
  pixels: the body runs at LR whatever the scale. Everything about that is speed. The
  quality of a ×3 or ×4 model on real content is not measured anywhere in this repo.

### On a mid-low card this is a 30 fps model at 1080p

Two independent estimates agree, and both are bands rather than numbers: the same card
read twice as fast in the sweep recorded in `summary.md` as it did here, months apart, so
every absolute figure spans a factor of two while every ratio survives intact. Inside that
band an Iris Xe or an M1 is over the whole 1080p60 budget with the decode still to pay
for, a GTX 1650 is at or over it, an RTX 3050 is borderline, and a mid card is inside.
**720p60 and 1080p30 are the honest targets for the low end**, and the frame is not the
thing to make cheaper — the tiles that did not change are.

### What changed outside this notebook

- `webexport.geometry`, `tiling`, `exact_tilings` and `plan_tiling` — the geometry is one
  shared rule now rather than an arithmetic expression repeated in every notebook, and
  `TARGET_KEPT_PX` is the tile this sweep chose.
- `webexport.export` takes a `batch`, and a `size` that may be (height, width). Neither
  was expressible before, and the two together are most of what was measured here.
- The benchmark page reports **ns / kept px** and **ns / frame px** and no longer reports
  frames per second, which could not compare two geometries. Its `runs / frame` divides
  by the batch, which it previously ignored.
- `TILE_STEP` is untouched at 128. It is what the page *compares* models at, and moving it
  would invalidate every published number without measuring anything.

### What is still owed

- **The same sweep on a mid-low card.** Every card but one in this notebook is a nominal
  TFLOPS figure and a ratio. The graphs are published and the page reports the right
  columns; it is an afternoon on the target hardware, and until it is done the mid-low
  conclusions are arithmetic rather than measurement.
- **Re-export the models at 320×180.** Everything published outside this notebook is still
  at the 128 step, including the model a client would actually run.
- **Train ×3 and ×4** on a dataset degraded at those factors. The cost is now known and
  the quality is not, which is the wrong way round for a decision.
- **Measure the skip rate at this tile.** 44% of tiles changed per frame at step 128; at
  640×360 there are nine tiles rather than forty and the fraction will be higher. The
  break-even above says tiling beats a whole-frame run below about 80% of tiles moving,
  and that is the number the whole tiling argument rests on.